# 03 架构家族、训练目标、MoE 和长上下文

目标：把 BERT/GPT/T5 这类架构差异、MLM/CLM/Seq2Seq 训练目标、attention mask、MoE 路由和滑动窗口长上下文放到小张量里练习。


## 1. 安装依赖


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


## 2. 三类主流 Transformer 架构

| 架构 | 代表 | Attention | 常见训练目标 | 适合任务 |
| --- | --- | --- | --- | --- |
| Encoder-only | BERT/RoBERTa | 双向 self-attention | MLM、分类/抽取微调 | 理解、分类、检索、NER |
| Decoder-only | GPT/LLaMA/Qwen | causal self-attention | next-token prediction | 生成、对话、代码、agent |
| Encoder-decoder | T5/BART | encoder 双向 + decoder causal + cross-attention | seq2seq denoising / translation | 翻译、摘要、结构化转换 |


In [ ]:
architectures = [
    {"name": "Encoder-only", "mask": "bidirectional", "objective": "masked language modeling", "example": "BERT"},
    {"name": "Decoder-only", "mask": "causal", "objective": "next-token prediction", "example": "GPT/LLaMA/Qwen"},
    {"name": "Encoder-decoder", "mask": "encoder bidirectional + decoder causal + cross attention", "objective": "sequence-to-sequence", "example": "T5/BART"},
]

for row in architectures:
    print("=" * 100)
    for key, value in row.items():
        print(f"{key:10s}: {value}")


## 3. Attention mask 对比

Encoder-only 可以看全句；decoder-only 只能看当前位置及之前；滑动窗口 causal mask 只能看有限历史。


In [ ]:
import torch

seq_len = 8
bidirectional_mask = torch.ones(seq_len, seq_len, dtype=torch.bool)
causal_mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))
window = 3
positions = torch.arange(seq_len)
sliding_window_mask = (positions[None, :] <= positions[:, None]) & ((positions[:, None] - positions[None, :]) < window)

print("bidirectional mask:")
print(bidirectional_mask.int())
print("\ncausal mask:")
print(causal_mask.int())
print(f"\nsliding window causal mask, window={window}:")
print(sliding_window_mask.int())


## 4. MLM 和 CLM 的 label 差异

BERT 类 MLM 只监督被 mask 的 token；GPT 类 CLM 监督每个有效位置的下一个 token。


In [ ]:
pad_id = 0
mask_id = 99
# 原句 token，假设 0 是 padding。
original = torch.tensor([[11, 12, 13, 14, 15, 0]])
masked_input = torch.tensor([[11, mask_id, 13, mask_id, 15, 0]])

mlm_labels = torch.full_like(original, -100)
mlm_labels[masked_input == mask_id] = original[masked_input == mask_id]

clm_input = original[:, :-1]
clm_labels = original[:, 1:].clone()
clm_labels[clm_labels == pad_id] = -100

print("MLM input :", masked_input)
print("MLM labels:", mlm_labels)
print("\nCLM input :", clm_input)
print("CLM labels:", clm_labels)


## 5. Encoder-decoder cross-attention 形状

Seq2Seq 模型的 decoder 不只看自己已生成的 token，还通过 cross-attention 看 encoder 输出。


In [ ]:
import math

batch = 2
src_len = 5
tgt_len = 3
hidden = 12
heads = 3
head_dim = hidden // heads

encoder_hidden = torch.randn(batch, src_len, hidden)
decoder_hidden = torch.randn(batch, tgt_len, hidden)

q = decoder_hidden.view(batch, tgt_len, heads, head_dim).transpose(1, 2)
k = encoder_hidden.view(batch, src_len, heads, head_dim).transpose(1, 2)
v = encoder_hidden.view(batch, src_len, heads, head_dim).transpose(1, 2)

cross_scores = q @ k.transpose(-2, -1) / math.sqrt(head_dim)
cross_weights = torch.softmax(cross_scores, dim=-1)
cross_context = cross_weights @ v

print("q from decoder:", tuple(q.shape))
print("k/v from encoder:", tuple(k.shape), tuple(v.shape))
print("cross attention scores:", tuple(cross_scores.shape))
print("cross context:", tuple(cross_context.shape))


## 6. MoE：top-k 路由和负载均衡

MoE 每个 token 只走部分 expert，因此总参数量可以很大，但每个 token 的激活参数较少。难点是路由、通信和负载均衡。


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
num_tokens = 12
hidden_size = 16
num_experts = 4
top_k = 2

hidden_states = torch.randn(num_tokens, hidden_size)
router = nn.Linear(hidden_size, num_experts, bias=False)
router_logits = router(hidden_states)
router_probs = F.softmax(router_logits, dim=-1)
top_probs, top_experts = torch.topk(router_probs, k=top_k, dim=-1)
normalized_top_probs = top_probs / top_probs.sum(dim=-1, keepdim=True)

expert_counts = torch.bincount(top_experts.reshape(-1), minlength=num_experts)
load_fraction = expert_counts.float() / expert_counts.sum()
prob_fraction = router_probs.mean(dim=0)
load_balance_loss = num_experts * torch.sum(load_fraction * prob_fraction)

print("top experts per token:")
print(top_experts)
print("normalized top probs:")
print(normalized_top_probs)
print("expert_counts:", expert_counts.tolist())
print("load_fraction:", load_fraction.tolist())
print("mean router prob:", prob_fraction.tolist())
print("toy load balance loss:", float(load_balance_loss))


## 7. MoE 输出的最小实现

这里每个 expert 是一个小 MLP。真实 MoE 还要考虑 token dispatch、capacity、all-to-all 通信和 expert parallel。


In [ ]:
experts = nn.ModuleList([
    nn.Sequential(nn.Linear(hidden_size, hidden_size * 2), nn.SiLU(), nn.Linear(hidden_size * 2, hidden_size))
    for _ in range(num_experts)
])

output = torch.zeros_like(hidden_states)
for token_index in range(num_tokens):
    for slot in range(top_k):
        expert_id = int(top_experts[token_index, slot])
        weight = normalized_top_probs[token_index, slot]
        expert_output = experts[expert_id](hidden_states[token_index:token_index + 1]).squeeze(0)
        output[token_index] += weight * expert_output

print("hidden_states:", tuple(hidden_states.shape))
print("moe output   :", tuple(output.shape))
print("mean abs output:", float(output.abs().mean()))


## 8. 长上下文方法怎么分类

- RoPE scaling / position interpolation：扩展位置编码可泛化范围，但质量要实测。
- Sliding window attention：每层只看局部历史，省显存和计算，但远距离依赖变弱。
- Global + local attention：少量全局 token 加局部窗口。
- Prefix/prompt cache：重复前缀直接复用 KV cache，降低 prefill 成本。
- RAG：不把全部知识塞进上下文，而是检索相关片段。


In [ ]:
def attention_score_elements(seq_len, mode="full", window=None):
    if mode == "full":
        return seq_len * seq_len
    if mode == "causal":
        return seq_len * (seq_len + 1) // 2
    if mode == "sliding":
        assert window is not None
        return sum(min(i + 1, window) for i in range(seq_len))
    raise ValueError(mode)

for length in [1024, 4096, 16384]:
    full = attention_score_elements(length, "full")
    causal = attention_score_elements(length, "causal")
    sliding = attention_score_elements(length, "sliding", window=1024)
    print("=" * 100)
    print("seq_len:", length)
    print("full score elements   :", full)
    print("causal score elements :", causal)
    print("sliding window=1024   :", sliding)
    print("sliding / causal      :", f"{sliding / causal:.3f}")


## 面试总结

- Encoder-only 看双向上下文，适合理解；decoder-only 只能看左侧上下文，适合生成；encoder-decoder 用 cross-attention 做输入到输出的转换。
- MLM 只监督 mask token；CLM 监督每个位置的 next token；Seq2Seq 监督 decoder 输出序列。
- attention mask 决定信息流，mask 错了训练目标和推理行为都会错。
- MoE 的核心是 token 路由到少数 expert，优势是扩大参数量，代价是路由、负载均衡和跨卡通信复杂。
- 长上下文不是只改一个 `max_length`，还涉及位置编码、attention 复杂度、KV cache、检索和缓存策略。
